#L-Trail: Reproducing Figure 1 (Pancreas dataset)

In this notebook, we replicate the conceptual framework of Figure 1 and the results for the pancreas dataset from Figure 2.


**Computational Requirements**<bt>

Please be aware that the benchmarking section involving scVelo is memory-intensive. Users who wish to focus exclusively on our novel algorithm may skip the scVelo cells to conserve memory and reduce execution time.

In [ ]:
#Installation of required dependencies

!pip install numpy==2.0.2 pandas==2.3.3 scipy==1.16.3 matplotlib==3.10.0 scikit-learn==1.6.1
!pip install anndata==0.12.10 scanpy==1.12.0
!pip install scvelo==0.3.4
!pip install leidenalg==0.11.0 igraph==1.0.0 gseapy==1.1.11
!pip install scikit-misc==0.5.2

In [ ]:
import scvelo as scv
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc
from scipy.stats import gaussian_kde, skew
import random
from scipy.stats import skew
from sklearn.neighbors import NearestNeighbors
import seaborn as sns
from scipy.spatial.distance import cosine

In [ ]:
SEED = 34

random.seed(SEED)
np.random.seed(SEED)
sc.settings.seed = SEED
scv.settings.seed = SEED

In [ ]:
#フォント設定
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans', 'Bitstream Vera Sans']

#figsize
plt.rcParams['font.size'] = 8          # 基本サイズ (8pt)
plt.rcParams['axes.titlesize'] = 10    # グラフタイトル (10pt)
plt.rcParams['axes.labelsize'] = 9     # 軸ラベル (9pt)
plt.rcParams['xtick.labelsize'] = 8    # 目盛り (8pt)
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8    # 凡例 (8pt)
plt.rcParams['figure.titlesize'] = 12  # 全体タイトル

#scanpy設定
#scanpy設定
sc.set_figure_params(
    scanpy=True,
    dpi=100,
    dpi_save=300,
    frameon=False,
    vector_friendly=True,
    fontsize=10,
    figsize=(6, 6),
    color_map='viridis'
    )

#関数の定義

##L-moment, L-trailの計算

In [ ]:
def _calc_l_moment_vector(data: np.ndarray) -> np.ndarray:
    """
    Estimate the transition direction of a cell population by calculating the L-moment
    (specifically L-skewness, tau3) along each dimension of the input data.

    This function calculates Probability Weighted Moments (PWM) and linearly transforms
    them into L-moments. The resulting L-skewness indicates the geometric asymmetry
    ("comet tail") of the distribution, which is used to define the L-Trail vector.

    Parameters
    ----------
    data : np.ndarray
        Input data array representing the coordinates of a cell population in a
        multi-dimensional space (e.g., PCA coordinates).
        Shape should be (n_cells, n_dimensions).

    Returns
    -------
    vec : np.ndarray
        The estimated directional vector (L-Flow) representing the transition direction.
        Shape is (n_dimensions,).
    """
    n_cells = data.shape[0]
    n_dims = data.shape[1]

    # Return a zero vector if the number of cells is insufficient for robust calculation
    if n_cells < 30:
        return np.zeros(n_dims)

    # 1. Order Statistics
    # Sort the data along each dimension: X(1) <= X(2) <= ... <= X(n)
    sorted_data = np.sort(data, axis=0)

    # ----------------------------------------------------------------------
    # 2. Probability Weighted Moments (PWM) Calculation
    # Formula: b_r = (1/n) * sum( x_i * weight_i )
    # ----------------------------------------------------------------------
    idx = np.arange(1, n_cells + 1)

    # b0: Equivalent to the standard mean
    b0 = np.mean(sorted_data, axis=0)

    # b1: Weighting by (i - 1) / (n - 1)
    w1 = (idx - 1) / (n_cells - 1)
    b1 = np.mean(sorted_data * w1[:, None], axis=0)

    # b2: Weighting by (i - 1)(i - 2) / ((n - 1)(n - 2))
    w2 = ((idx - 1) * (idx - 2)) / ((n_cells - 1) * (n_cells - 2))
    b2 = np.mean(sorted_data * w2[:, None], axis=0)

    # ----------------------------------------------------------------------
    # 3. Derivation of L-moments from PWMs
    # ----------------------------------------------------------------------
    # lamda2 (L-scale): Represents dispersion
    lamda2 = 2 * b1 - b0

    # lamda3 (L-3rd moment): Represents asymmetry / skewness
    lamda3 = 6 * b2 - 6 * b1 + b0

    # ----------------------------------------------------------------------
    # 4. L-skewness (Tau-3) Calculation
    # ----------------------------------------------------------------------
    # Tau3 = lamda3 / lamda2
    # Suppress warnings and handle division by zero or very small L-scale values
    with np.errstate(divide='ignore', invalid='ignore'):
        tau3 = np.divide(lamda3, lamda2)
        # Avoid computational artifacts where dispersion is essentially zero
        tau3[lamda2 < 1e-6] = 0.0

    # ----------------------------------------------------------------------
    # 5. Definition of L-Flow (Directional Vector)
    # ----------------------------------------------------------------------
    # The vector direction is determined by the negative skewness (-tau3),
    # scaled back by the dispersion (lamda2) to represent the magnitude of the tail.
    vec = -tau3 * lamda2

    return vec

##高次元空間(PCA)におけるクラスターの重心とベクトルを計算


In [ ]:
def _calc_high_dim_vector(
    data: np.ndarray,
    method: str = 'lmoment',
    scale_skew_std: bool = True
) -> tuple:
    """
    Calculate the centroid and directional vector for a given cluster in high-dimensional space.

    Parameters
    ----------
    data : np.ndarray
        High-dimensional data array for a specific cluster (e.g., PCA coordinates).
        Shape should be (n_cells, n_dimensions).
    method : str, optional (default: 'lmoment')
        Method used to calculate the directional vector representing the trajectory.
        Available options are:
        - 'lmoment': Vector based on L-moments (recommended).
        - 'pearson': Vector based on Pearson's second coefficient of skewness.
        - 'skew': Vector based on standard Fisher-Pearson coefficient of skewness.
    scale_skew_std : bool, optional (default: True)
        Whether to scale the standard skewness vector by the standard deviation.
        Only applicable when `method='skew'`.

    Returns
    -------
    mean_pos : np.ndarray
        The centroid (mean position) of the cluster.
    vec : np.ndarray
        The calculated directional vector indicating the trajectory of the cluster.
    """
    mean_pos = np.mean(data, axis=0)

    if method == 'pearson':
        # Utilizing the difference between mean and median
        # Based on Pearson's second coefficient of skewness
        median_pos = np.median(data, axis=0)
        vec = 3 * (median_pos - mean_pos)

    elif method == 'skew':
        # Standard Fisher-Pearson coefficient of skewness
        sk = skew(data, axis=0)
        if scale_skew_std:
            std = np.std(data, axis=0)
            vec = -sk * std
        else:
            vec = -sk

    elif method == 'lmoment':
        # L-moment based vector calculation
        vec = _calc_l_moment_vector(data)

    else:
        raise ValueError(f"Unknown method '{method}'. Expected 'lmoment', 'pearson', or 'skew'.")

    return mean_pos, vec

##有意差検定:ブートストラップ

In [ ]:
def _test_significance_high_dim(
    subset: np.ndarray,
    observed_magnitude: float,
    method: str,
    n_boot: int = 1000
) -> float:
    """
    Perform a permutation test to calculate the statistical significance (p-value) of the directional vector.

    This function generates a null distribution by centering the data and randomly
    flipping the signs of the coordinates, testing the null hypothesis that the
    observed directional magnitude is due to chance.

    Parameters
    ----------
    subset : np.ndarray
        High-dimensional data array for a specific cluster (e.g., PCA coordinates).
        Shape should be (n_cells, n_dimensions).
    observed_magnitude : float
        The magnitude (Euclidean length) of the observed directional vector.
    method : str
        Method used to calculate the directional vector (e.g., 'lmoment', 'pearson', 'skew').
        This is passed internally to `_calc_high_dim_vector`.
    n_boot : int, optional (default: 1000)
        Number of permutations (bootstrap iterations) to perform for generating
        the null distribution.

    Returns
    -------
    p_val : float
        The calculated empirical p-value representing the statistical significance
        of the directional vector.
    """
    # Centering the data to set up the null hypothesis
    centered = subset - np.mean(subset, axis=0)
    n_cells = centered.shape[0]
    null_mags = []

    for _ in range(n_boot):
        # Generate data with randomly flipped signs (1 or -1)
        signs = np.random.choice([-1, 1], size=(n_cells, 1))
        shuffled = centered * signs

        try:
            # Calculate the vector for the shuffled null data
            _, vec_null = _calc_high_dim_vector(shuffled, method=method)
            mag_null = np.sqrt(np.sum(vec_null**2))
            null_mags.append(mag_null)
        except Exception:
            # Append 0.0 if calculation fails for extreme random cases
            null_mags.append(0.0)

    null_mags = np.array(null_mags)

    # Calculate the empirical p-value
    p_val = (np.sum(null_mags >= observed_magnitude) + 1) / (n_boot + 1)

    return p_val

##L-Trailの可視化関数

In [ ]:
def plot_ltrail(adata,
               groupby,
               basis='X_pca',
               use_rep='X_pca',
               n_pcs=30,
               k=30,
               method='lmoment',
               scale=5.0,
               p_threshold=0.05,
               n_boot=100,
               min_cells=20,
               legend_loc='right margin',
               figsize=(8, 6),
               dot_size=None,
               title=None,
               ax=None,
               alpha=0.3,
               show=True,
               save=None):
    """
    Project the L-Trail calculated in a high-dimensional space onto an arbitrary low-dimensional embedding.

    This function calculates the trajectory vector for each cluster in the high-dimensional PCA space
    and maps it to a low-dimensional visualization space (e.g., UMAP or PCA) using a k-Nearest
    Neighbors (k-NN) approach.

    Parameters
    ----------
    adata : AnnData
        Annotated data matrix containing the single-cell dataset.
    groupby : str
        Key in `adata.obs` defining the clusters for vector calculation.
    basis : str, optional (default: 'X_pca')
        Key in `adata.obsm` for the low-dimensional embedding used for visualization.
    use_rep : str, optional (default: 'X_pca')
        Key in `adata.obsm` for the high-dimensional space used for vector calculation.
    n_pcs : int, optional (default: 30)
        Number of principal components to use from the high-dimensional space.
    k : int, optional (default: 30)
        Number of nearest neighbors used for the projection mapping.
    method : str, optional (default: 'lmoment')
        Method used for calculating the directional vector ('lmoment', 'pearson', 'skew').
    scale : float, optional (default: 5.0)
        Scaling factor for the length of the drawn vectors.
    p_threshold : float or None, optional (default: 0.05)
        P-value threshold for the permutation test. Vectors with a p-value above this
        threshold will not be drawn. Set to None to bypass the significance test.
    n_boot : int, optional (default: 100)
        Number of bootstrap iterations for the permutation test.
    min_cells : int, optional (default: 20)
        Minimum number of cells required in a cluster to calculate its vector.
    legend_loc : str, optional (default: 'right margin')
        Location of the legend in the scatter plot.
    figsize : tuple, optional (default: (8, 6))
        Width and height of the figure in inches.
    dot_size : int, optional (default: None)
        Point size for the background scatter plot.
    title : str, optional (default: None)
        Title of the plot.
    ax : matplotlib.axes.Axes, optional (default: None)
        A matplotlib axes object to draw the plot on. If None, a new figure is created.
    alpha : float, optional (default: 0.3)
        Alpha blending value for the background scatter plot points.
    show : bool, optional (default: True)
        Whether to display the figure immediately.
    save : str, optional (default: None)
        Filename to save the figure (e.g., 'figure.pdf').

    Returns
    -------
    fig : matplotlib.figure.Figure or None
        Returns the matplotlib Figure object if `show=False`, else returns None.
    """

    show_plot = False
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
        show_plot = True
    else:
        fig = ax.get_figure()

    # Check data consistency
    if use_rep not in adata.obsm.keys():
        raise ValueError(f"Calculation space '{use_rep}' not found in adata.obsm.")
    if basis not in adata.obsm.keys():
        raise ValueError(f"Visualization space '{basis}' not found in adata.obsm.")

    # 1. High-dimensional computation space
    X_high = adata.obsm[use_rep][:, :n_pcs]

    # 2. Low-dimensional visualization space
    X_vis = adata.obsm[basis][:, :2]

    # Build k-NN graph for projection
    nbrs = NearestNeighbors(n_neighbors=k, algorithm='auto').fit(X_high)

    groups = adata.obs[groupby]
    unique_groups = groups.unique()

    if title is None:
        sig_txt = f", p<{p_threshold}" if p_threshold else ""
        title = f"L-Trail Estimation ({method}, scale={scale})\nSpace: {use_rep} -> {basis}{sig_txt}"

    scanpy_basis = basis.replace('X_', '') if basis.startswith('X_') else basis

    # Background scatter plot
    sc.pl.embedding(adata,
                    basis=scanpy_basis,
                    color=groupby,
                    legend_loc=legend_loc,
                    size=dot_size,
                    title=title,
                    ax=ax,
                    show=False,
                    frameon=True,
                    alpha=alpha)

    print(f"--- Processing Information ---")
    print(f"Method       : {method}")
    print(f"Input Space  : {use_rep} (Top {n_pcs} PCs)")
    print(f"Output Space : {basis} (Projected via k-NN)")

    # Calculate vectors for each cluster
    quiver_X, quiver_Y = [], [] # Start points
    quiver_U, quiver_V = [], [] # Vector components (dx, dy)

    for group in unique_groups:

        mask = (groups == group).values
        subset_high = X_high[mask]

        if len(subset_high) < min_cells:
            continue

        try:
            # Step A: Calculate vector in high-dimensional space
            mean_high, vec_high = _calc_high_dim_vector(subset_high, method=method)
            magnitude = np.sqrt(np.sum(vec_high**2))

            # Step B: Significance test
            if p_threshold is not None:
                if magnitude < 1e-6:
                    continue

                p_val = _test_significance_high_dim(subset_high,
                                                    observed_magnitude=magnitude,
                                                    method=method,
                                                    n_boot=n_boot)
                if p_val >= p_threshold:
                    continue # Skip if not statistically significant

            # Step C: Projection (k-NN Mapping)
            # Find the projected start point
            dists_s, idxs_s = nbrs.kneighbors([mean_high])
            start_2d = np.mean(X_vis[idxs_s[0]], axis=0)

            # Find the projected end point
            future_high = mean_high + (vec_high * scale)
            dists_e, idxs_e = nbrs.kneighbors([future_high])
            end_2d = np.mean(X_vis[idxs_e[0]], axis=0)

            # Calculate 2D vector components
            dx = end_2d[0] - start_2d[0]
            dy = end_2d[1] - start_2d[1]

            # Remove negligible noisy vectors
            if np.sqrt(dx**2 + dy**2) < 0.01:
                continue

            # Append to drawing lists
            quiver_X.append(start_2d[0])
            quiver_Y.append(start_2d[1])
            quiver_U.append(dx)
            quiver_V.append(dy)

            # Highlight the start points of the vectors
            ax.scatter(start_2d[0],
                       start_2d[1],
                       c='#222222',
                       edgecolor='white',
                       linewidth=0.5,
                       s=15,
                       zorder=11
                       )

        except Exception as e:
            print(f"Skipped {group}: {e}")

    # Draw the vectors using matplotlib quiver
    if quiver_X:
        print(f"Drawing {len(quiver_X)} vectors...")
        ax.quiver(quiver_X,
                  quiver_Y,
                  quiver_U,
                  quiver_V,
                  angles='xy',
                  scale_units='xy',
                  scale=1,
                  color='#222222',
                  edgecolor='white',
                  linewidth=0.5,
                  headwidth=5,
                  headlength=6,
                  headaxislength=5,
                  zorder=12)
    else:
        print("Warning: No vectors to draw! Check thresholds or data.")

    # Handle saving and display
    if save is not None:
        fig.savefig(save, bbox_inches='tight')
        print(f"Saved figure to: {save}")

    if show:
        plt.show()
        return None

    return fig

##VelocityとL-Trailのcos類似度を計測する関数

In [ ]:
def calc_velocity_ltrail_similarity(
    adata,
    groupby: str,
    use_rep: str = 'X_pca',
    vel_rep: str = 'velocity_pca',
    method: str = 'lmoment',
    n_pca: int = 30,
    min_cells: int = 20
) -> pd.DataFrame:
    """
    Calculate the cosine similarity between RNA velocity vectors and L-Trail vectors for each cluster.

    This function computes the mean RNA velocity vector for a given cluster and
    compares it with the L-Trail directional vector calculated in the same high-dimensional space
    using cosine similarity.

    Parameters
    ----------
    adata : AnnData
        Annotated data matrix containing the single-cell dataset.
    groupby : str
        Key in `adata.obs` defining the clusters for vector calculation.
    use_rep : str, optional (default: 'X_pca')
        Key in `adata.obsm` for the high-dimensional space used for calculation.
    vel_rep : str, optional (default: 'velocity_pca')
        Key in `adata.obsm` containing the RNA velocity vectors projected in the high-dimensional space.
    method : str, optional (default: 'lmoment')
        Method used for calculating the L-Trail directional vector ('lmoment', 'pearson', 'skew').
    n_pca : int, optional (default: 30)
        Number of principal components to use from the high-dimensional space.
    min_cells : int, optional (default: 20)
        Minimum number of cells required in a cluster to calculate its vector.

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the cluster name, cosine similarity score, and calculation status (Note),
        sorted in descending order of cosine similarity.
    """
    if use_rep not in adata.obsm.keys():
        raise ValueError(f"'{use_rep}' not found in adata.obsm.")
    if vel_rep not in adata.obsm.keys():
        raise ValueError(f"'{vel_rep}' not found in adata.obsm. Did you run scVelo/Velocity first?")

    X_high = adata.obsm[use_rep][:, :n_pca]
    V_high = adata.obsm[vel_rep][:, :n_pca]  # Velocity vectors

    groups = adata.obs[groupby]
    unique_groups = groups.unique()

    results = []

    for group in unique_groups:
        mask = (groups == group)
        subset_high = X_high[mask]
        subset_vel = V_high[mask]

        if len(subset_high) < min_cells:
            continue

        try:
            # 1. Calculate L-Trail vector
            _, vec_ltrail = _calc_high_dim_vector(subset_high, method=method)

            # 2. Calculate the mean RNA velocity vector for the cluster
            vec_vel = np.mean(subset_vel, axis=0)

            # Avoid zero/noise vectors
            mag_ltrail = np.linalg.norm(vec_ltrail)
            mag_vel = np.linalg.norm(vec_vel)

            if mag_ltrail < 1e-6 or mag_vel < 1e-6:
                results.append({
                    'Cluster': group,
                    'Cos_Similarity': np.nan,
                    'Note': 'zero vector'
                })
                continue

            # 3. Calculate Cosine Similarity
            # Note: scipy.spatial.distance.cosine returns the cosine distance (1 - similarity)
            sim = 1.0 - cosine(vec_vel, vec_ltrail)

            results.append({
                'Cluster': group,
                'Cos_Similarity': sim,
                'Note': 'Success'
            })

        except Exception as e:
            results.append({
                'Cluster': group,
                'Cos_Similarity': np.nan,
                'Note': f'Error: {e}'
            })

    # Compile results into a DataFrame and sort
    df_results = pd.DataFrame(results).sort_values(by='Cos_Similarity', ascending=False)

    return df_results

##クラスターをグリッド分けして計算する関数

In [ ]:
def calc_grid_similarity(
    adata,
    use_rep: str = 'X_pca',
    vel_rep: str = 'velocity_pca',
    n_pca: int = 30,
    method: str = 'lmoment',
    grid_size: int = 15,
    min_cells: int = 20
) -> pd.DataFrame:
    """
    Calculate the cosine similarity between RNA velocity and L-Trail vectors across a spatial grid.

    This function divides the top two dimensions of the specified representation space
    into a uniform grid, and calculates the cosine similarity between the mean RNA velocity
    and the L-Trail directional vector for the cells within each grid bin.

    Parameters
    ----------
    adata : AnnData
        Annotated data matrix containing the single-cell dataset.
    use_rep : str, optional (default: 'X_pca')
        Key in `adata.obsm` for the high-dimensional space used for grid definition and calculation.
    vel_rep : str, optional (default: 'velocity_pca')
        Key in `adata.obsm` containing the RNA velocity vectors.
    n_pca : int, optional (default: 30)
        Number of principal components to use for the high-dimensional vector calculation.
    method : str, optional (default: 'lmoment')
        Method used for calculating the L-Trail directional vector ('lmoment', 'pearson', 'skew').
    grid_size : int, optional (default: 15)
        Number of bins (grid cells) along each of the two dimensions.
    min_cells : int, optional (default: 20)
        Minimum number of cells required in a grid bin to perform the calculation.

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the Grid ID, cell count, center coordinates of the grid,
        and the cosine similarity score, sorted in descending order of similarity.
    """
    if use_rep not in adata.obsm.keys():
        raise ValueError(f"'{use_rep}' not found in adata.obsm.")
    if vel_rep not in adata.obsm.keys():
        raise ValueError(f"'{vel_rep}' not found in adata.obsm. Did you run scVelo/Velocity first?")

    # 1. Define the spatial grid using the top 2 components
    coords = adata.obsm[use_rep][:, :2]

    # Create evenly spaced bins
    x_min, x_max = coords[:, 0].min(), coords[:, 0].max()
    y_min, y_max = coords[:, 1].min(), coords[:, 1].max()
    x_bins = np.linspace(x_min, x_max, grid_size + 1)
    y_bins = np.linspace(y_min, y_max, grid_size + 1)

    # Determine which grid bin each cell belongs to
    x_indices = np.digitize(coords[:, 0], x_bins)
    y_indices = np.digitize(coords[:, 1], y_bins)
    grid_ids = np.array([f"{x}_{y}" for x, y in zip(x_indices, y_indices)])

    # 2. Retrieve high-dimensional vectors
    X_high = adata.obsm[use_rep][:, :n_pca]
    V_high = adata.obsm[vel_rep][:, :n_pca]
    unique_grids = np.unique(grid_ids)

    results = []

    # Calculate similarity for each grid bin
    for grid_id in unique_grids:
        mask = (grid_ids == grid_id)
        subset_high = X_high[mask]
        subset_vel = V_high[mask]

        # Skip grid bins with insufficient cells
        if len(subset_high) < min_cells:
            continue

        try:
            # Calculate L-Trail vector
            _, vec_ltrail = _calc_high_dim_vector(subset_high, method=method)

            # Calculate mean RNA velocity vector
            vec_vel = np.nanmean(subset_vel, axis=0)

            # Calculate magnitudes to filter out zero/noise vectors
            mag_ltrail = np.linalg.norm(vec_ltrail)
            mag_vel = np.linalg.norm(vec_vel)

            if mag_ltrail < 1e-6 or mag_vel < 1e-6:
                continue

            # Calculate Cosine Similarity
            sim = 1.0 - cosine(vec_vel, vec_ltrail)

            # Store the center coordinates of the grid bin
            center_x = np.mean(coords[mask, 0])
            center_y = np.mean(coords[mask, 1])

            results.append({
                'Grid_ID': grid_id,
                'Cell_Count': len(subset_high),
                'Center_X': center_x,
                'Center_Y': center_y,
                'Cos_Similarity': sim
            })

        except Exception:
            # Silently skip grids where calculation fails (e.g., math errors on extreme data)
            pass

    # Compile results into a DataFrame
    df_results = pd.DataFrame(results)
    if not df_results.empty:
        df_results = df_results.sort_values(by='Cos_Similarity', ascending=False)

    return df_results

##グリッド毎のcos類似度を可視化

In [ ]:
def plot_grid_similarity_map(
    adata,
    df_grid_sim: pd.DataFrame,
    basis: str = 'X_pca',
    title: str = None,
    figsize: tuple = (8, 6),
    show: bool = True
):
    """
    Visualize the grid-based cosine similarity map on a low-dimensional embedding space.

    This function plots the background cells and overlays the center of each spatial grid.
    The color of the grid points represents the cosine similarity between RNA velocity
    and L-Trail vectors, and the size of the points is scaled by the number of cells in that grid.

    Parameters
    ----------
    adata : AnnData
        Annotated data matrix containing the single-cell dataset.
    df_grid_sim : pd.DataFrame
        DataFrame containing grid similarity results, typically generated by `calc_grid_similarity`.
    basis : str, optional (default: 'X_pca')
        Key in `adata.obsm` used for plotting the background cells.
    title : str, optional (default: None)
        Title of the plot. If None, a default descriptive title is used.
    figsize : tuple, optional (default: (8, 6))
        Width and height of the figure in inches.
    show : bool, optional (default: True)
        Whether to display the figure immediately. If False, returns the Axes object.

    Returns
    -------
    ax : matplotlib.axes.Axes or None
        Returns the matplotlib Axes object if `show=False`, else returns None.
    """
    if df_grid_sim.empty:
        print("Error: No data to plot.")
        return

    fig, ax = plt.subplots(figsize=figsize)

    # 1. Plot background cells
    if basis in adata.obsm:
        coords = adata.obsm[basis]
        ax.scatter(coords[:, 0],
                   coords[:, 1],
                   c='lightgray',
                   s=10,
                   alpha=0.3,
                   label='Background Cells',
                   zorder=1)
    else:
        print(f"Warning: '{basis}' not found in adata.obsm. Background cells skipped.")

    # 2. Plot grid centers
    # Scale point sizes based on the cell count within each grid
    sizes = df_grid_sim['Cell_Count'] * 3

    scatter = ax.scatter(df_grid_sim['Center_X'],
                         df_grid_sim['Center_Y'],
                         c=df_grid_sim['Cos_Similarity'],
                         s=sizes,
                         cmap='coolwarm',
                         vmin=-1.0,
                         vmax=1.0,
                         edgecolors='black',
                         linewidths=0.5,
                         zorder=5)

    # 3. Aesthetics and Colorbar
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Cosine Similarity (Velocity vs L-Trail)')

    # Set title
    if title:
        ax.set_title(title)
    else:
        ax.set_title("Grid-based Cosine Similarity Map\nRed: Aligned (>0), Blue: Opposing (<0)")

    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.grid(False)

    if show:
        plt.show()
        return None

    return ax

##k-NNのクラスター毎にcos類似度を測定


In [ ]:
def calc_knn_similarity(
    adata,
    use_rep: str = 'X_pca',
    vel_rep: str = 'velocity_pca',
    n_pcs: int = 30,
    method: str = 'lmoment',
    k: int = 50,
    n_anchors: int = 1000,
    random_state: int = 34
) -> pd.DataFrame:
    """
    Calculate the local cosine similarity between RNA velocity and L-Trail vectors using a k-NN approach.

    This function samples random anchor cells, identifies their k-nearest neighbors in a
    high-dimensional space, and compares the mean RNA velocity vector of the neighborhood
    with the local L-Trail directional vector using cosine similarity.

    Parameters
    ----------
    adata : AnnData
        Annotated data matrix containing the single-cell dataset.
    use_rep : str, optional (default: 'X_pca')
        Key in `adata.obsm` for the high-dimensional space used for calculation.
    vel_rep : str, optional (default: 'velocity_pca')
        Key in `adata.obsm` containing the RNA velocity vectors.
    n_pcs : int, optional (default: 30)
        Number of principal components to use from the high-dimensional space.
    method : str, optional (default: 'lmoment')
        Method used for calculating the L-Trail directional vector ('lmoment', 'pearson', 'skew').
    k : int, optional (default: 50)
        Number of nearest neighbors to define the local neighborhood. A relatively large `k`
        is recommended to maintain the stability and accuracy of the L-Trail calculation.
    n_anchors : int, optional (default: 1000)
        Number of random anchor cells to sample for the evaluation. If None, all cells are used.
    random_state : int, optional (default: 34)
        Random seed for sampling the anchor cells to ensure reproducibility.

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the Anchor Index, 2D Center coordinates, and the
        calculated Cosine Similarity for each valid anchor neighborhood.
    """
    if use_rep not in adata.obsm.keys():
        raise ValueError(f"'{use_rep}' not found in adata.obsm.")
    if vel_rep not in adata.obsm.keys():
        raise ValueError(f"'{vel_rep}' not found in adata.obsm. Did you run scVelo/Velocity first?")

    # 1. Retrieve high-dimensional coordinates and velocity vectors
    X_high = adata.obsm[use_rep][:, :n_pcs]
    V_high = adata.obsm[vel_rep][:, :n_pcs]

    # 2D coordinates for visualization purposes (usually PC1 and PC2)
    coords_2d = adata.obsm[use_rep][:, :2]

    n_cells = X_high.shape[0]

    # 2. Sample anchor cells
    np.random.seed(random_state)
    if n_anchors is not None and n_anchors < n_cells:
        anchor_indices = np.random.choice(n_cells, n_anchors, replace=False)
    else:
        anchor_indices = np.arange(n_cells)

    # 3. Build the k-NN graph in the high-dimensional space
    print(f"Building k-NN graph (k={k}) in {use_rep} space...")
    nbrs = NearestNeighbors(n_neighbors=k, algorithm='auto').fit(X_high)
    _, indices = nbrs.kneighbors(X_high[anchor_indices])

    results = []

    # 4. Calculate similarity for each anchor's neighborhood
    print(f"Calculating similarities for {len(anchor_indices)} anchor points...")
    for i, anchor_idx in enumerate(anchor_indices):
        neighbors_idx = indices[i]  # Indices of the k nearest neighbors including the anchor

        subset_high = X_high[neighbors_idx]
        subset_vel = V_high[neighbors_idx]

        try:
            # Calculate L-Trail vector for the local neighborhood
            _, vec_ltrail = _calc_high_dim_vector(subset_high, method=method)

            # Calculate the mean RNA velocity vector for the local neighborhood
            vec_vel = np.nanmean(subset_vel, axis=0)

            # Calculate magnitudes to filter out zero/noise vectors
            mag_ltrail = np.linalg.norm(vec_ltrail)
            mag_vel = np.linalg.norm(vec_vel)

            if mag_ltrail < 1e-6 or mag_vel < 1e-6:
                continue

            # Calculate Cosine Similarity
            sim = 1.0 - cosine(vec_vel, vec_ltrail)

            results.append({
                'Anchor_Index': anchor_idx,
                'Center_X': coords_2d[anchor_idx, 0],
                'Center_Y': coords_2d[anchor_idx, 1],
                'Cos_Similarity': sim,
            })

        except Exception:
            # Silently skip if calculation fails
            pass

    df_results = pd.DataFrame(results)
    print("Done.")

    return df_results

##k-NN結果を撒布図上で描画する

In [ ]:
def plot_knn_similarity(
    adata,
    df_results: pd.DataFrame,
    basis: str = 'X_pca',
    title: str = None,
    figsize: tuple = (8, 6),
    show: bool = True
):
    """
    Visualize the k-NN based local cosine similarity on a low-dimensional embedding space.

    This function plots the background cells from the single-cell dataset and overlays
    the anchor cells evaluated in `calc_knn_similarity`. The anchor cells are colored
    according to their cosine similarity between RNA velocity and L-Trail vectors.

    Parameters
    ----------
    adata : AnnData
        Annotated data matrix containing the single-cell dataset.
    df_results : pd.DataFrame
        DataFrame containing the local similarity results, typically generated by `calc_knn_similarity`.
    basis : str, optional (default: 'X_pca')
        Key in `adata.obsm` used for plotting the background cells.
    title : str, optional (default: None)
        Title of the plot. If None, a default descriptive title is used.
    figsize : tuple, optional (default: (8, 6))
        Width and height of the figure in inches.
    show : bool, optional (default: True)
        Whether to display the figure immediately. If False, returns the Axes object.

    Returns
    -------
    ax : matplotlib.axes.Axes or None
        Returns the matplotlib Axes object if `show=False`, else returns None.
    """
    # Check for empty data
    if df_results.empty:
        print("Warning: No data to plot in plot_knn_similarity.")
        return

    fig, ax = plt.subplots(figsize=figsize)

    # 1. Plot background cells
    if basis in adata.obsm:
        coords_bg = adata.obsm[basis][:, :2]
        ax.scatter(coords_bg[:, 0],
                   coords_bg[:, 1],
                   c='lightgray',
                   alpha=0.2,
                   s=10,
                   zorder=1)
    else:
        print(f"Warning: '{basis}' not found in adata.obsm. Background cells skipped.")

    # 2. Plot evaluated anchor cells colored by cosine similarity
    scat = ax.scatter(
        df_results['Center_X'],
        df_results['Center_Y'],
        c=df_results['Cos_Similarity'],
        cmap='coolwarm',  # Blue (opposing) - White (orthogonal) - Red (aligned)
        vmin=-1.0,
        vmax=1.0,
        s=30,
        edgecolor='black',
        linewidth=0.5,
        zorder=2
    )

    # Add colorbar
    plt.colorbar(scat, ax=ax, label='Cosine Similarity (Velocity vs L-Trail)')

    # Set title
    if title:
        ax.set_title(title)
    else:
        ax.set_title("k-NN based Vector Similarity (Manifold-aware)")

    # Set axis labels
    scanpy_basis = basis.replace('X_', '') if basis.startswith('X_') else basis
    ax.set_xlabel(f"{scanpy_basis}1")
    ax.set_ylabel(f"{scanpy_basis}2")

    if show:
        plt.show()
        return None

    return ax

##k-NNで計算した結果をboxplotにまとめる

In [ ]:
def boxplot_similarity(
    adata,
    df_results: pd.DataFrame,
    groupby: str = 'clusters',
    title: str = None,
    figsize: tuple = (10, 6),
    ylim: tuple = None,
    show: bool = True
):
    """
    Draw a publication-ready combined Violin and Box plot of cosine similarities.

    This function visualizes the distribution of cosine similarities between RNA velocity
    and L-Trail vectors across different cell clusters. It automatically calculates and
    displays the sample size (n) and median value for each cluster on the x-axis labels.

    Parameters
    ----------
    adata : AnnData
        Annotated data matrix containing the single-cell dataset.
    df_results : pd.DataFrame
        DataFrame containing the local similarity results, typically generated by `calc_knn_similarity`.
        Must contain the 'Anchor_Index' column.
    groupby : str, optional (default: 'clusters')
        Key in `adata.obs` defining the clusters to group the similarities by.
    title : str, optional (default: None)
        Title of the plot.
    figsize : tuple, optional (default: (10, 6))
        Width and height of the figure in inches.
    ylim : tuple, optional (default: None)
        Y-axis limits. For cosine similarity, (-1.0, 1.0) or (-1.1, 1.1) is recommended.
    show : bool, optional (default: True)
        Whether to display the figure immediately. If False, returns the Axes object.

    Returns
    -------
    ax : matplotlib.axes.Axes or None
        Returns the matplotlib Axes object if `show=False`, else returns None.
    """
    # 1. Set plot style
    sns.set_style("ticks")

    df_plot = df_results.copy()

    # Data validation
    if 'Anchor_Index' not in df_plot.columns:
        print("Error: 'Anchor_Index' column not found in df_results.")
        return
    if groupby not in adata.obs:
        print(f"Error: '{groupby}' not found in adata.obs.")
        return

    # Map cluster information from adata to the results DataFrame
    try:
        indices = df_plot['Anchor_Index'].values.astype(int)
        cluster_labels = adata.obs[groupby].iloc[indices].values
        df_plot['Cluster'] = cluster_labels
    except Exception as e:
        print(f"Error mapping clusters: {e}")
        return

    # Calculate summary statistics to order the plot by median (descending)
    summary_stats = df_plot.groupby('Cluster')['Cos_Similarity'].agg(['count', 'median']).sort_values(by='median', ascending=False)
    median_order = summary_stats.index

    if len(median_order) == 0:
        print("Warning: No data to plot.")
        return

    fig, ax = plt.subplots(figsize=figsize)

    # Violin plot (background distribution)
    sns.violinplot(
        data=df_plot,
        x='Cluster',
        y='Cos_Similarity',
        order=median_order,
        inner=None,
        color='lightgray',
        linewidth=0,
        alpha=0.4,
        ax=ax
    )

    # Box plot (foreground statistics)
    sns.boxplot(
        data=df_plot,
        x='Cluster',
        y='Cos_Similarity',
        order=median_order,
        width=0.25,
        color='#4C72B0',
        showfliers=False,
        linewidth=1.2,
        ax=ax
    )

    # Draw reference line at 0 (orthogonal vectors)
    ax.axhline(0, color='#333333', linestyle='--', linewidth=1, alpha=0.8)

    # Set title and axis labels
    if title:
        ax.set_title(title, fontsize=14, fontweight='bold', pad=15)

    ax.set_ylabel('Cosine similarity', fontsize=12, fontweight='bold')
    ax.set_xlabel('Cell groups', fontsize=12, fontweight='bold')

    # Append n-count and median to X-axis labels
    new_labels = []
    for cluster in median_order:
        count = int(summary_stats.loc[cluster, 'count'])
        median_val = summary_stats.loc[cluster, 'median']
        new_labels.append(f"{cluster}\n(n={count})\nMed: {median_val:.2f}")

    ax.set_xticks(range(len(median_order)))
    ax.set_xticklabels(new_labels, rotation=45, ha='right', fontsize=10)
    ax.tick_params(axis='y', labelsize=10)

    # Remove top and right spines
    sns.despine(ax=ax)

    if ylim:
        ax.set_ylim(ylim)

    plt.tight_layout()

    if show:
        plt.show()
        return None

    return ax

#Figure1 B



In [ ]:
# 1D L-moment calculation function
def calc_l_moments_vector_1d(data):
    """
    Computes a directional vector based on L-skewness and L-scale.
    This represents the "L-flow" in a single dimension.
    """
    n = len(data)
    sorted_data = np.sort(data)
    idx = np.arange(1, n + 1)

    # Calculate Probability Weighted Moments (PWMs)
    b0 = np.mean(sorted_data)
    w1 = (idx - 1) / (n - 1)
    b1 = np.mean(sorted_data * w1)
    w2 = (idx - 1) * (idx - 2) / ((n - 1) * (n - 2))
    b2 = np.mean(sorted_data * w2)

    # L-moments
    lambda2 = 2 * b1 - b0
    lambda3 = 6 * b2 - 6 * b1 + b0

    if lambda2 == 0: return 0
    tau3 = lambda3 / lambda2

    # Return the directional vector (Proposed)
    return -tau3 * lambda2

# --- Simulation Setup ---
np.random.seed(34)
n_cells = 200

# Generate main population (Gamma distribution)
main_pop = np.random.gamma(shape=2.0, scale=1.0, size=n_cells)
x = main_pop

# Introduce extreme outliers (simulating technical artifacts)
outliers = np.random.uniform(-10, -8, size=5)
x_corrupted = np.concatenate([x, outliers])

# A. Conventional Approach (Pearson Skewness * SD)
mean_val = np.mean(x_corrupted)
sk_val = skew(x_corrupted)
std_val = np.std(x_corrupted)
vec_pearson = -sk_val * std_val

# B. Proposed Approach (L-moment based)
vec_lflow = calc_l_moments_vector_1d(x_corrupted)

# --- Visualization: Comparative Analysis of Robustness ---
fig, ax = plt.subplots(figsize=(8, 4))

# Histogram (Underlying Distribution)
ax.hist(x_corrupted,
        bins=30,
        density=True,
        alpha=0.5,
        color='gray',
        label='Cell Population (with noise)')

# Plot outliers for visual emphasis
ax.scatter(outliers,
           np.zeros_like(outliers) + 0.01,
           color='red',
           marker='x',
           linewidth=1.5,
           label='Technical Outliers')

# Vertical positioning for the arrows
baseline_y = 0.15

# Arrow A: Conventional Method (Pearson-based)
# Demonstrating high sensitivity to outliers
ax.arrow(mean_val,
         baseline_y,
         vec_pearson,
         0,
         head_width=0.02,
         head_length=0.5,
         fc='red',
         ec='red',
         width=0.005,
         label=f'Conventional (Pearson): Skew={sk_val:.2f}')

# Arrow B: Proposed Method (L-moment/L-flow)
# Demonstrating robustness and stability
ax.arrow(mean_val,
         baseline_y + 0.05,
         vec_lflow,
         0,
         head_width=0.02,
         head_length=0.5,
         fc='blue',
         ec='blue',
         width=0.005,
         label='Proposed (L-moment based)')

# --- Figure Formatting ---
ax.set_title("Sensitivity Comparison: L-moments vs. Conventional Moments", fontsize=14, fontweight='bold')
ax.set_xlabel("Latent Space / Gene Expression (e.g., PC1)", fontsize=12)
ax.set_yticks([]) # Hide Y-axis as it's for density visualization
ax.legend(loc='upper left', fontsize=9, frameon=True)
ax.grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

#2. Data Acquisiton



Load the pancreas dataset using the scVelo built-in library.


In [ ]:
adata_pc = scv.datasets.pancreas()
adata_pc_velo = adata_pc.copy()

#3. scVelo Benchmark (Stochastioc Mode)

Compute and visualize L-Trail trajectories within the same embedding space as the RNA velocity.

チュートリアル通りのパイプラインで解析する

In [ ]:
adata_pc_velo = adata_pc.copy()
scv.settings.verbosity = 3
scv.settings.presenter_view = True
scv.set_figure_params('scvelo')

In [ ]:
scv.pp.filter_genes(adata_pc_velo,
                    min_shared_counts=20)
sc.pp.normalize_total(adata_pc_velo)
sc.pp.log1p(adata_pc_velo)
sc.pp.filter_genes_dispersion(adata_pc_velo, n_top_genes=2000)

scv.pp.moments(adata_pc_velo, n_pcs=30, n_neighbors=30)
scv.tl.velocity(adata_pc_velo, mode='stochastic')
scv.tl.velocity_graph(adata_pc_velo)

In [ ]:
#明示的にpca, umapを計算
sc.tl.pca(adata_pc_velo, svd_solver='arpack')
sc.tl.umap(adata_pc_velo)

In [ ]:
scv.tl.velocity_graph(adata_pc_velo)

細胞数の確認

In [ ]:
pc_cellcounts = adata_pc_velo.obs['clusters'].value_counts()
labels = [f"{cluster}\n(n={count})" for cluster, count in zip(pc_cellcounts.index, pc_cellcounts.values)]

fig, ax = plt.subplots(figsize=(15, 6))

sns.barplot(
    x=pc_cellcounts.index,
    y=pc_cellcounts.values,
    order=pc_cellcounts.index,  # <--- ここを追加！順番を固定します
    color='#4C72B0',
    edgecolor='black',
    linewidth=1,
    ax=ax
)

# これで棒の順番とラベルの順番が一致するので、正しく上書きされます
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=11)
ax.tick_params(axis='y', labelsize=11)

ax.set_xlabel('Pancreas Clusters', fontsize=14, labelpad=10)
ax.set_ylabel('Number of Cells', fontsize=14, labelpad=10)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.spines['left'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)

ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle='--', color='lightgrey', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
scv.pl.velocity_embedding_stream(adata_pc_velo,
                                 basis='umap',
                                 color='clusters',
                                 figsize=(8, 6),
                                 legend_loc='right margin',
                                 title='Pancreas RNA Velocity (Stochastic, umap)',
                                 show=False,
                                 )

scv.pl.velocity_embedding_stream(adata_pc_velo,
                                 basis='pca',
                                 color='clusters',
                                 figsize=(8, 6),
                                 legend_loc='right margin',
                                 title='Pancreas RNA Velocity (Stochastic, pca)',
                                 show=False
                                 )



#4. L-Trail Implementation

Compute and visualize L-Trail trajectories within the same embedding space as the RNA velocity.<br>

Note:


In [ ]:
plot_ltrail(adata_pc_velo,
            groupby='clusters',
            basis='X_pca',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, pca)'
            )

plot_ltrail(adata_pc_velo,
            groupby='clusters',
            basis='X_umap',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, pca)'
            )

#cos類似度

In [ ]:
calc_velocity_ltrail_similarity(adata_pc_velo,
                                groupby='clusters',
                                method='lmoment',
                                use_rep='X_pca',
                                vel_rep='velocity_pca',
                                )

##グリッドでcos類似度を測定




In [ ]:
##k-NNでクラスタリングを行いcos類似度を測定

In [ ]:
result_knn_cos_lmoment = calc_knn_similarity(adata_pc_velo,
                                             use_rep='X_pca',
                                             vel_rep='velocity_pca',
                                             n_pcs=30,
                                             method='lmoment',
                                             k=50,
                                             n_anchors=1000,
                                             )

plot_knn_similarity(adata_pc_velo,
                    result_knn_cos_lmoment,
                    basis='X_pca',
                    title='Pancreas cos similarity (L-Moment vs Velocity)',
                    figsize=(8, 6),
                    show=True
                    )

boxplot_similarity(adata_pc_velo,
                        result_knn_cos_lmoment,
                        groupby='clusters',
                        figsize=(8, 6),
                        ylim=(-1, 1),
                        title='Pancreas cos similarity (L-Moment vs Velocity)'
                        )

#Supply

In [ ]:
pc_velo_03 = adata_pc_velo.copy()
pc_velo_05 = adata_pc_velo.copy()
pc_velo_1 = adata_pc_velo.copy()
sc.tl.leiden(pc_velo_03, resolution=0.3)
sc.tl.leiden(pc_velo_05, resolution=0.5)
sc.tl.leiden(pc_velo_1, resolution=1)


plot_ltrail(pc_velo_03,
            groupby='leiden',
            basis='X_umap',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, umap, resolution=0.3)'
            )

plot_ltrail(pc_velo_05,
            groupby='leiden',
            basis='X_umap',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, umap, resolution=0.5)'
            )

plot_ltrail(pc_velo_1,
            groupby='leiden',
            basis='X_umap',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, umap, resolution=1)'
            )



In [ ]:
plot_ltrail(pc_velo_03,
            groupby='leiden',
            basis='X_umap',
            use_rep='X_pca',
            method='pearson',
            scale=30,
            p_threshold=0.05,
            title='Pancreas pearson Cluster(scale=30, p<0.05, umap, resolution=0.3)'
            )

plot_ltrail(pc_velo_05,
            groupby='leiden',
            basis='X_umap',
            use_rep='X_pca',
            method='pearson',
            scale=30,
            p_threshold=0.05,
            title='Pancreas pearson Cluster(scale=30, p<0.05, umap, resolution=0.5)'
            )

plot_ltrail(pc_velo_1,
            groupby='leiden',
            basis='X_umap',
            use_rep='X_pca',
            method='pearson',
            scale=30,
            p_threshold=0.05,
            title='Pancreas pearson Cluster(scale=30, p<0.05, umap, resolution=1)'
            )

In [ ]:
plot_ltrail(pc_velo_03,
            groupby='leiden',
            basis='X_pca',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, pca, resolution=0.3)'
            )

plot_ltrail(pc_velo_05,
            groupby='leiden',
            basis='X_pca',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, pca, resolution=0.5)'
            )

plot_ltrail(pc_velo_1,
            groupby='leiden',
            basis='X_pca',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, pca, resolution=1)'
            )



In [ ]:
plot_ltrail(pc_velo_03,
            groupby='leiden',
            basis='X_pca',
            use_rep='X_pca',
            method='pearson',
            scale=30,
            p_threshold=0.05,
            title='Pancreas pearson Cluster(scale=30, p<0.05, pca, resolution=0.3)'
            )

plot_ltrail(pc_velo_05,
            groupby='leiden',
            basis='X_pca',
            use_rep='X_pca',
            method='pearson',
            scale=30,
            p_threshold=0.05,
            title='Pancreas pearson Cluster(scale=30, p<0.05, pca, resolution=0.5)'
            )

plot_ltrail(pc_velo_1,
            groupby='leiden',
            basis='X_pca',
            use_rep='X_pca',
            method='pearson',
            scale=30,
            p_threshold=0.05,
            title='Pancreas pearson Cluster(scale=30, p<0.05, pca, resolution=1)'
            )